# 08 · PyTorch benchmark architecture inspection

Read source snippets from v19.1/v17 and compare coded model classes. No heavy training is triggered. The labels `Mamba` and `DIINA` in these benchmark scripts should not be confused with the manuscript's architectural claims.

In [ ]:
from pathlib import Path
import json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project():
    candidates=[Path.cwd(), Path.cwd()/"project_inputs", Path.cwd().parent/"project_inputs", Path("/mnt/data/diina_reproducibility/project_inputs"), Path("/mnt/data")]
    for p in candidates:
        if (p/"raw_results.csv").exists(): return p
    raise FileNotFoundError("Place notebook with project_inputs/ or set cwd to extracted project directory")
PROJECT=find_project()
df=pd.read_csv(PROJECT/"raw_results.csv")
summary=json.loads((PROJECT/"summary.json").read_text())
print("Project files:",PROJECT,"| telemetry shape:",df.shape,"| columns:",list(df.columns))
assert len(df)==1500 and set(df.columns)=={"seed","step","loss","fast_usage","slow_usage","entropy"}

In [ ]:
def class_excerpt(path,class_name):
    text=(PROJECT/path).read_text(); start=text.find("class "+class_name)
    if start<0: return "class not found"
    nextpos=text.find("\nclass ",start+1)
    return text[start: nextpos if nextpos>=0 else min(len(text),start+1800)]
for name in ["TransformerModel","MambaBaseline","DIINA"]:
    print("\n###",name,"(v19.1)"); print(class_excerpt("DIINA_v19_1_LongDistance.py",name)[:1500])

In [ ]:
import ast
source=(PROJECT/"DIINA_v19_1_LongDistance.py").read_text()
# Static source evidence; AST parse confirms the supplied v19.1 file is syntactically valid.
try:
    ast.parse(source); print("v19.1 Python source parses successfully")
except SyntaxError as e: print("Source syntax issue:",e)
print("\nArchitectural reading:")
print("TransformerBaseline: embedding + TransformerEncoder + query-position linear classifier")
print("MambaBaseline: embedding + nn.GRU + last-state classifier (GRU proxy, not Mamba/SSM)")
print("DIINA: embedding + nn.GRU + sigmoid Linear gate; state multiplied by (1-gate)")
print("Manuscript: separate Fast and high-capacity Slow streams coordinated by Linguistic Inhibition Gate (LIG).")

**Important distinction:** This package contains exact log parsing and rule-level generator ports, educational statistical curve models, and Python benchmark source inspection. The benchmark's DIINA is a single gated GRU; it does not instantiate the manuscript's dual-stream Fast/Slow architecture. v17 uses its own Transformer and gated-GRU DIINA variant.